## PCA + Metadata

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from scipy.stats import mannwhitneyu
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

from scipy.stats import ttest_ind

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [2]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df_dataset = pd.read_parquet(DATASET_PATH)
df_dataset.shape

(108, 360)

In [3]:
#  Age, Gender and condition
# Create a dictionary where keys are the metric names (e.g., 'degree')
NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
]

NODE_FEATURES = [
    c for c in df_dataset.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]
df_dataset = df_dataset.rename(columns={
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status"
})

In [4]:
# descarta colunas sem informação (constantes ou duplicadas), se houver
DROP_REDUNDANT = True

_raw = df_dataset[NODE_FEATURES]
constant_cols = [c for c in NODE_FEATURES if _raw[c].nunique(dropna=True) <= 1]
_dup = _raw.T.duplicated()
duplicate_cols = [c for c, d in zip(NODE_FEATURES, _dup) if d and c not in constant_cols]

if DROP_REDUNDANT:
    NODE_FEATURES = [c for c in NODE_FEATURES if c not in constant_cols and c not in duplicate_cols]

print(f"Constantes: {len(constant_cols)} | duplicadas: {len(duplicate_cols)} | usadas no PCA: {len(NODE_FEATURES)}")
print("Tipos constantes:", sorted({c.rsplit("_", 1)[0] for c in constant_cols}))
print("Tipos duplicados:", sorted({c.rsplit("_", 1)[0] for c in duplicate_cols}))

Constantes: 0 | duplicadas: 0 | usadas no PCA: 212
Tipos constantes: []
Tipos duplicados: []


In [5]:

X = df_dataset[NODE_FEATURES]
meta = df_dataset[["condition", "age", "gender"]]


imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
pca = PCA(n_components=2)

X_imputed = imputer.fit_transform(X)
X_scaled = scaler.fit_transform(X_imputed)
X_pca = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
df_pca = pd.concat([df_pca, meta.reset_index(drop=True)], axis=1)
df_pca

,PC1,PC2,condition,age,gender
0,-17.559520,4.519905,EC,20-25,2
1,-13.609292,-1.020230,EO,20-25,2
2,1.881172,-1.036262,EC,20-25,2
3,-7.297157,-1.414439,EO,20-25,2
4,6.562239,-0.118392,EC,25-30,1
...,...,...,...,...,...
103,-7.885076,-2.121277,EO,20-25,2
104,2.374983,-0.363282,EC,30-35,2
105,-3.613465,-1.109669,EO,30-35,2
106,7.113544,0.026071,EC,20-25,2


In [6]:

color_vars = ["condition", "age", "gender"]

for var in color_vars:
    fig = px.scatter(
        df_pca,
        x="PC1",
        y="PC2",
        color=var,
        title=f"PCA colored by {var}",
        opacity=0.8,
        template="plotly_dark"
    )
    fig.show()


A projeção por PCA revelou ampla sobreposição entre os grupos EC e EO nos dois primeiros componentes principais, não indicando separação visual evidente entre as condições com base nas métricas de rede selecionadas. No plano PC1–PC2 também não há agrupamentos visuais por gênero. Por faixa etária, os adultos mais velhos tendem a ficar do lado de PC1 mais baixo, com bastante sobreposição; essa tendência é testada abaixo com um valor por participante e é significativa. Foram identificados alguns participantes com escores extremos, sugerindo perfis de conectividade distintos que devem ser investigados individualmente.

In [7]:
pca.explained_variance_ratio_, pca.explained_variance_ratio_.sum()

(array([0.52849916, 0.06562737]), np.float64(0.594126529049813))

### Cargas dos componentes

Quais métricas produzem a variação capturada por PC1 e PC2.

In [8]:
loadings = pd.DataFrame(pca.components_.T, index=NODE_FEATURES, columns=["PC1", "PC2"])
loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(10).round(3)

,PC1,PC2
clustering_P3,0.093,0.021
clustering_P4,0.093,0.026
clustering_P1,0.093,0.019
clustering_P2,0.093,0.019
clustering_PO3,0.093,0.026
clustering_P6,0.093,0.028
clustering_PO4,0.093,0.026
clustering_POz,0.093,0.025
clustering_P5,0.093,0.025
clustering_Pz,0.093,0.020


In [9]:
loadings.reindex(loadings["PC2"].abs().sort_values(ascending=False).index).head(10).round(3)

,PC1,PC2
betweenness_P7,-0.038,0.229
betweenness_CP1,-0.040,0.227
betweenness_C1,-0.041,0.226
betweenness_Cz,-0.035,0.224
betweenness_FC2,-0.039,0.220
betweenness_P1,-0.046,0.212
betweenness_CP4,-0.044,0.207
betweenness_Oz,-0.049,0.205
betweenness_P8,-0.043,0.199
betweenness_C2,-0.048,0.196


In [10]:
_lt = loadings[["PC1", "PC2"]].assign(tipo=[c.rsplit("_", 1)[0] for c in loadings.index])

# cada componente tem norma 1: a soma das cargas ao quadrado de um tipo é a fração do componente formada por ele
loadings_by_type = _lt.groupby("tipo").agg(
    n_variaveis=("PC1", "size"),
    PC1_contribuicao=("PC1", lambda s: (s ** 2).sum()),
    PC2_contribuicao=("PC2", lambda s: (s ** 2).sum()),
    PC1_carga_media=("PC1", "mean"),
    PC1_frac_positivas=("PC1", lambda s: (s > 0).mean()),
).sort_values("PC1_contribuicao", ascending=False)

fig = px.bar(
    loadings_by_type.reset_index().melt(
        id_vars="tipo", value_vars=["PC1_contribuicao", "PC2_contribuicao"],
        var_name="componente", value_name="contribuicao",
    ),
    x="tipo", y="contribuicao", color="componente", barmode="group",
    title="Contribuição de cada tipo de métrica para PC1 e PC2",
    labels={"tipo": "Métrica", "contribuicao": "Soma das cargas²", "componente": "Componente"},
)
fig.show()

loadings_by_type.round(3)

,n_variaveis,PC1_contribuicao,PC2_contribuicao,PC1_carga_media,PC1_frac_positivas
tipo,,,,,
clustering,53,0.435,0.032,0.091,1.000
degree,53,0.403,0.040,0.087,1.000
betweenness,53,0.092,0.825,-0.040,0.000
hub_frequency,53,0.071,0.103,-0.034,0.019


### PC1 e idade (um valor por participante)

Cada participante aparece em duas linhas (EO e EC), que não são independentes. Por isso os testes usam um valor por pessoa:

1. **Efeito da idade:** média de PC1 entre EO e EC de cada participante, comparada entre adultos jovens e adultos mais velhos (Mann-Whitney + correlação rank-biserial). EO e EC isolados aparecem como descrição.
2. **Interação idade × condição:** diferença EO − EC de PC1 de cada participante, comparada entre os grupos. Só com essa diferença significativa dá para dizer que o efeito da idade é maior numa condição do que na outra.

Grupos seguem o desenho do LEMON: adultos jovens 20–40 anos, adultos mais velhos 55–80. O sinal de PC1 é arbitrário; a leitura vem das cargas acima.

In [11]:
def rank_biserial(a, b):
    u = mannwhitneyu(a, b, alternative="two-sided").statistic
    return 2 * u / (len(a) * len(b)) - 1


df_age = df_pca.assign(subject_id=df_dataset["subject_id"].to_numpy())
df_age["age_group"] = np.where(df_age["age"].str.split("-").str[0].astype(int) < 55, "jovem", "idoso")

per_subject = (
    df_age.pivot_table(index=["subject_id", "age_group"], columns="condition", values="PC1")
    .reset_index()
)
per_subject["PC1_media"] = per_subject[["EO", "EC"]].mean(axis=1)
per_subject["PC1_EO_menos_EC"] = per_subject["EO"] - per_subject["EC"]

age_tests = []
for label, col in [
    ("Idade (média EO/EC)", "PC1_media"),
    ("Idade, só EO (descritivo)", "EO"),
    ("Idade, só EC (descritivo)", "EC"),
    ("Interação idade × condição (EO − EC)", "PC1_EO_menos_EC"),
]:
    jov = per_subject.loc[per_subject["age_group"] == "jovem", col].dropna()
    ido = per_subject.loc[per_subject["age_group"] == "idoso", col].dropna()
    age_tests.append({
        "teste": label,
        "n_jovem": len(jov),
        "n_idoso": len(ido),
        "mediana_jovem": jov.median(),
        "mediana_idoso": ido.median(),
        "p_valor": mannwhitneyu(ido, jov, alternative="two-sided").pvalue,
        "rank_biserial_idoso_vs_jovem": rank_biserial(ido, jov),
    })

pd.DataFrame(age_tests).round(4)

,teste,n_jovem,n_idoso,mediana_jovem,mediana_idoso,p_valor,rank_biserial_idoso_vs_jovem
0,Idade (média EO/EC),35,19,4.1666,-2.2863,0.0035,-0.4857
1,"Idade, só EO (descritivo)",35,19,1.6409,-7.0378,0.0006,-0.5729
2,"Idade, só EC (descritivo)",35,19,6.0361,1.5879,0.0483,-0.3293
3,Interação idade × condição (EO − EC),35,19,-1.3864,-6.2756,0.0101,-0.4286


In [12]:
n_jov = int((per_subject["age_group"] == "jovem").sum())
n_ido = int((per_subject["age_group"] == "idoso").sum())

age_table = (
    pd.DataFrame(age_tests)
    .loc[:, ["teste", "mediana_jovem", "mediana_idoso", "p_valor", "rank_biserial_idoso_vs_jovem"]]
    .rename(columns={
        "teste": "Teste",
        "mediana_jovem": f"Adultos jovens (n = {n_jov}), mediana",
        "mediana_idoso": f"Adultos mais velhos (n = {n_ido}), mediana",
        "p_valor": "p",
        "rank_biserial_idoso_vs_jovem": "Rank-biserial (mais velhos vs jovens)",
    })
    .set_index("Teste")
)
age_table.round(2).assign(p=age_table["p"].round(4))

,"Adultos jovens (n = 35), mediana","Adultos mais velhos (n = 19), mediana",p,Rank-biserial (mais velhos vs jovens)
Teste,,,,
Idade (média EO/EC),4.17,-2.29,0.0035,-0.49
"Idade, só EO (descritivo)",1.64,-7.04,0.0006,-0.57
"Idade, só EC (descritivo)",6.04,1.59,0.0483,-0.33
Interação idade × condição (EO − EC),-1.39,-6.28,0.0101,-0.43


In [13]:
from pathlib import Path

AGE_LABELS = {"jovem": "Adultos jovens", "idoso": "Adultos mais velhos"}
MEASURE_LABELS = {
    "PC1_media": "PC1 média de EO e EC",
    "PC1_EO_menos_EC": "Diferença de PC1 (EO − EC)",
}

plot_df = per_subject.melt(
    id_vars=["subject_id", "age_group"],
    value_vars=list(MEASURE_LABELS),
    var_name="medida",
    value_name="valor",
)
plot_df["medida"] = plot_df["medida"].map(MEASURE_LABELS)
plot_df["age_group"] = plot_df["age_group"].map(AGE_LABELS)

# fundo claro e letras grandes: mesma versão no notebook e no TCC
fig_tcc = px.box(
    plot_df,
    x="age_group",
    y="valor",
    color="age_group",
    facet_col="medida",
    facet_col_spacing=0.12,
    points="all",
    category_orders={"age_group": list(AGE_LABELS.values()), "medida": list(MEASURE_LABELS.values())},
    labels={"age_group": "Grupo etário"},
    template="plotly_white",
)
fig_tcc.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font_size=26))
fig_tcc.update_traces(pointpos=0, jitter=0.35, marker=dict(size=12, opacity=0.8), line_width=3)
fig_tcc.update_yaxes(matches=None, showticklabels=True, title_font_size=26, tickfont_size=22,
                     zeroline=True, zerolinecolor="gray", zerolinewidth=2)
fig_tcc.update_yaxes(title_text="PC1", col=1)
fig_tcc.update_yaxes(title_text="Diferença de PC1", col=2)
fig_tcc.update_xaxes(title_font_size=26, tickfont_size=22)
fig_tcc.update_layout(
    showlegend=False,
    font=dict(size=22, color="black"),
    width=1400,
    height=700,
    margin=dict(l=100, r=40, t=70, b=90),
)
fig_tcc.show()

out_png = Path("../../figures/fig_pc1_idade.png")
out_png.parent.mkdir(parents=True, exist_ok=True)
try:
    fig_tcc.write_image(out_png, scale=3)
    print(f"Salvo: {out_png.resolve()}")
except ValueError as err:
    # kernel iniciado antes de instalar o kaleido: reinicie o kernel para exportar PNG
    out_html = out_png.with_suffix(".html")
    fig_tcc.write_html(out_html)
    print(f"PNG não exportado ({err.__class__.__name__}). HTML: {out_html.resolve()}")

Salvo: C:\Users\Anne\OneDrive\Documentos\Google Drive\Ciência de Dados\MBA USP\TCC\Código\TCC\figures\fig_pc1_idade.png


Com um valor por participante (35 adultos jovens, 19 adultos mais velhos), PC1 médio entre EO e EC foi menor nos adultos mais velhos (mediana −2,29 vs 4,17; Mann-Whitney p = 0,004; rank-biserial = −0,49, efeito grande). Como PC1 tem cargas positivas de grau ponderado e clustering em todos os eletrodos, os adultos mais velhos apresentam, em média, menor nível geral de sincronização.

A interação idade × condição também foi significativa: a diferença EO − EC de PC1 foi mais negativa nos adultos mais velhos (mediana −6,28 vs −1,39; p = 0,010; rank-biserial = −0,43). Ou seja, a queda de sincronização ao abrir os olhos é maior nos adultos mais velhos, o que explica por que a diferença entre grupos parece maior em EO (p < 0,001) do que em EC (p = 0,048). Os testes por condição isolada são descritivos; a afirmação de que o efeito da idade difere entre condições se apoia no teste de interação.